# micro-chat on Kaggle
Settings: **Accelerator = GPU (T4 x2 or P100)**, **Internet = On**, add a `HF_TOKEN` secret if you want to upload.

In [ ]:
!git clone https://github.com/12in-the-night/micro-chat.git /kaggle/working/micro-chat
%cd /kaggle/working/micro-chat

In [ ]:
!pip install -q -U "jax[cuda12]"
!pip install -q -r requirements.txt
!pip install -q -U huggingface_hub

In [ ]:
import jax
print(jax.devices())
assert len(jax.devices()) >= 1

In [ ]:
import yaml
p = "config.yaml"
c = yaml.safe_load(open(p))
ROOT = "/kaggle/working/micro-chat"
c["training"]["ckpt_dir"]       = f"{ROOT}/ckpts"
c["tokenizer"]["path"]         = f"{ROOT}/tokenizer.json"
c["postprocess"]["input_ckpt"]  = f"{ROOT}/ckpts/ckpt_final.pkl"
c["postprocess"]["output_dir"]  = f"{ROOT}/ckpts_compressed"
c["postprocess"]["export_path"] = f"{ROOT}/pi_model.npz"
yaml.safe_dump(c, open(p, "w"))
print("patched", ROOT)

In [ ]:
# optional: verify the install before spending GPU hours
!python -m pytest -q tests

In [ ]:
!python main.py --action tokenizer

In [ ]:
# To continue an interrupted run, set `resume: ./ckpts/ckpt_XXXXXXX.pkl` in config.yaml first.
!python main.py

In [ ]:
!python main.py --action sample --prompt "Once upon a time"

In [ ]:
!python main.py --action full

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

REPO = "rgdfbd/micro-chat-pi"
ROOT = "/kaggle/working/micro-chat"
UPLOAD_PICKLED_CHECKPOINTS = False   # pickles are not safe to load from strangers

api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
api.create_repo(repo_id=REPO, repo_type="model", exist_ok=True)

for name in ("pi_model.npz", "tokenizer.json", "config.yaml",
             "infer.py", "pi-need.txt"):
    fp = f"{ROOT}/{name}"
    print(f"[{'OK' if os.path.exists(fp) else 'MISSING':7s}] {name}")
    if os.path.exists(fp):
        api.upload_file(path_or_fileobj=fp, path_in_repo=name,
                        repo_id=REPO, repo_type="model")

if UPLOAD_PICKLED_CHECKPOINTS:
    api.upload_folder(folder_path=f"{ROOT}/ckpts_compressed",
                      path_in_repo="ckpts_compressed", repo_id=REPO, repo_type="model")
    api.upload_file(path_or_fileobj=f"{ROOT}/ckpts/ckpt_final.pkl",
                    path_in_repo="ckpt_final.pkl", repo_id=REPO, repo_type="model")

print(f"https://huggingface.co/{REPO}")